# Bronze Layer - Raw Data Ingestion

This notebook loads the latest Smart City batch from the raw layer,
performs basic validation, and writes the datasets to the Bronze layer.

## 1. Import Libraries

In [0]:
import os
from datetime import datetime

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

from utils.config import *

## 2. Load Configuration


In [0]:
PROJECT_ROOT = os.path.abspath("../../")

RAW_FOLDER = os.path.join(
    PROJECT_ROOT,
    RAW_PATH
)

BRONZE_FOLDER = os.path.join(
    PROJECT_ROOT,
    BRONZE_PATH
)

print("Project Root :", PROJECT_ROOT)
print("Raw Folder   :", RAW_FOLDER)
print("Bronze Folder:", BRONZE_FOLDER)

## 3. Locate Latest Batch

In [0]:
batch_folders = sorted(
    [
        folder
        for folder in os.listdir(RAW_FOLDER)
        if os.path.isdir(
            os.path.join(RAW_FOLDER, folder)
        )
    ]
)

LATEST_BATCH = batch_folders[-1]

CURRENT_BATCH = os.path.join(
    RAW_FOLDER,
    LATEST_BATCH
)

print("Latest Batch :", LATEST_BATCH)
print("Batch Path   :", CURRENT_BATCH)

## 4. Load Raw Datasets

In [0]:
BUS_FILE = os.path.join(
    CURRENT_BATCH,
    "bus_gps_clean.json"
)

EMERGENCY_FILE = os.path.join(
    CURRENT_BATCH,
    "emergency_clean.csv"
)

bus_df = spark.read.option(
    "multiline",
    "true"
).json(BUS_FILE)

emergency_df = spark.read.option(
    "header",
    "true"
).csv(EMERGENCY_FILE)

print("Datasets loaded successfully.")

## 5. Validate Input Files

In [0]:
print("Bus Records       :", bus_df.count())
print("Emergency Records :", emergency_df.count())

In [0]:
bus_df.printSchema()

In [0]:
emergency_df.printSchema()

## 6. Add Metadata

bus dataset

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bus_df = (
    bus_df
    .withColumn("batch_id", lit(LATEST_BATCH))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("bus_simulator"))
)

In [0]:
bus_df.show(5, truncate=False)

emergency dataset

In [0]:
emergency_df = (
    emergency_df
    .withColumn("batch_id", lit(LATEST_BATCH))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("emergency_simulator"))
)

In [0]:
emergency_df.show(5, truncate=False)

## 7. Create Bronze Layer

In [0]:
os.makedirs(BRONZE_FOLDER, exist_ok=True)

print(BRONZE_FOLDER)

## 8. Write Bronze Layer

In [0]:
bus_df.write \
    .mode("overwrite") \
    .parquet(
        os.path.join(
            BRONZE_FOLDER,
            "bus_gps"
        )
    )

In [0]:
emergency_df.write \
    .mode("overwrite") \
    .parquet(
        os.path.join(
            BRONZE_FOLDER,
            "emergency"
        )
    )

## 9. Verify Bronze

In [0]:
bus_bronze = spark.read.parquet(
    os.path.join(
        BRONZE_FOLDER,
        "bus_gps"
    )
)

emergency_bronze = spark.read.parquet(
    os.path.join(
        BRONZE_FOLDER,
        "emergency"
    )
)

print("Bus Bronze Records:", bus_bronze.count())
print("Emergency Bronze Records:", emergency_bronze.count())

## 10. Execution Summary

In [0]:
print("=" * 60)

print("Bronze Layer Completed")

print("=" * 60)

print("Batch ID :", LATEST_BATCH)

print("Bus Records :", bus_df.count())

print("Emergency Records :", emergency_df.count())

print("Status : SUCCESS")

print("=" * 60)